# SC-10-Account-Abstraction - ERC-4337

[<< DAO Governance](SC-9-DAO-Governance.ipynb) | [LLM Assisted >>](SC-11-LLM-Assisted.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**account abstraction** (ERC-4337)
2. Creer des **UserOperations**
3. Implementer un **Smart Account** simple
4. Comprendre les **Paymasters**

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-7](SC-9-DAO-Governance.ipynb) completes
- Comprendre le modele de compte Ethereum (EOA vs contrats)
- Notions sur ERC-4337 (optionnel, couvert dans le notebook)

### Duree estimee : 50 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [ ]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
import sys, os
from web3 import Web3
import solcx

# Import forge_helper pour compiler avec Foundry (supporte @openzeppelin, @account-abstraction)
sys.path.insert(0, os.path.abspath(".."))
from forge_helper import forge_compile, forge_compile_and_deploy

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity (via solcx, sans imports externes)."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


---

## 1. Concepts ERC-4337

ERC-4337 permet de separer la logique des comptes des wallets externes.

| Composant | Description |
|-----------|-------------|
| **UserOperation** | Transaction utilisateur (comme transfer, swap) |
| **EntryPoint** | Contrat singleton qui traite les UserOps |
| **Smart Account** | Wallet intelligent (logique metier en contrat) |
| **Paymaster** | Sponsor de frais de gas |
| **Bundler** | Empaquete les UserOps en batch |

In [ ]:
# UserOperation structure
USER_OP_STRUCT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

struct UserOperation {
    address sender;
    uint256 nonce;
    bytes initCode;
    bytes callData;
    uint256 callGasLimit;
    uint256 verificationGasLimit;
    uint256 preVerificationGas;
    uint256 maxFeePerGas;
    uint256 maxPriorityFeePerGas;
    bytes paymasterAndData;
    bytes signature;
}
'''

print("Structure UserOperation (ERC-4337):")
print(USER_OP_STRUCT)
print("Chaque champ controle un aspect de la transaction :")
print("  sender          = adresse du Smart Account")
print("  nonce           = protection contre le replay")
print("  callData        = la transaction a executer")
print("  signature       = preuve d'autorisation")
print("  paymasterAndData = donnees du paymaster (si gas sponsorise)")

---

## 2. Smart Account Simple

Un smart account est un contrat qui peut executer des transactions.

In [2]:
# Simple Smart Account (ERC-4337)
# Ce contrat herite de BaseAccount (account-abstraction SDK).
# Il necessite un EntryPoint deploye pour fonctionner en production.
# Ici on compile avec Foundry pour verifier la validite du code.

SIMPLE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@account-abstraction/contracts/core/BaseAccount.sol";
import "@account-abstraction/contracts/core/Helpers.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

contract SimpleAccount is BaseAccount {
    using ECDSA for bytes32;

    address public owner;
    IEntryPoint private immutable _entryPoint;

    event SimpleAccountInitialized(IEntryPoint indexed entryPoint, address indexed owner);

    constructor(address anOwner, IEntryPoint anEntryPoint) {
        owner = anOwner;
        _entryPoint = anEntryPoint;
    }

    function entryPoint() public view override returns (IEntryPoint) {
        return _entryPoint;
    }

    // Verification de signature
    function _validateSignature(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash
    ) internal virtual override returns (uint256 validationData) {
        bytes32 hash = MessageHashUtils.toEthSignedMessageHash(userOpHash);
        if (owner != ECDSA.recover(hash, userOp.signature)) {
            return SIG_VALIDATION_FAILED;
        }
        return SIG_VALIDATION_SUCCESS;
    }

    // Autoriser owner en plus de l EntryPoint
    function _requireForExecute() internal view override {
        require(
            msg.sender == address(_entryPoint) || msg.sender == owner,
            "only entry point or owner"
        );
    }

    receive() external payable {}
}
'''

# Compilation avec Foundry (resout les imports @account-abstraction et @openzeppelin)
abi, bytecode = forge_compile(SIMPLE_ACCOUNT, "SimpleAccount")
print(f"Compilation reussie ! ABI: {len(abi)} fonctions/events")
print(f"Bytecode: {len(bytecode)} caracteres")
print()

# Afficher les fonctions publiques du contrat compile
print("Fonctions et events du SimpleAccount:")
for item in abi:
    if item.get('type') == 'function':
        inputs = ', '.join(f"{i['type']} {i['name']}" for i in item.get('inputs', []))
        print(f"  function {item['name']}({inputs})")
    elif item.get('type') == 'event':
        print(f"  event {item['name']}")

# Note: le deploiement necessite un EntryPoint reel.
print()
print("Note: deploiement impossible sans EntryPoint reel.")
print("Voir la version standalone ci-dessous pour la demo deployable.")


### Version standalone deployable

Le contrat ci-dessus depend d'un EntryPoint deploye. Voici une version
autonome qui illustre les memes concepts (owner, execution, validation)
sans dependances externes.

In [ ]:
# Version standalone du Smart Account (sans dependances ERC-4337)
# Deploiement reel sur anvil

STANDALONE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title StandaloneSmartAccount
/// @notice Version simplifiee d un smart account ERC-4337 pour demonstration.
/// Illustre: owner, execution de transactions, batch, nonce.
contract StandaloneSmartAccount {
    address public owner;
    uint256 public nonce;

    event Executed(address indexed dest, uint256 value, bytes data);
    event OwnerChanged(address indexed oldOwner, address indexed newOwner);

    modifier onlyOwner() {
        require(msg.sender == owner, "not owner");
        _;
    }

    constructor(address _owner) {
        owner = _owner;
    }

    // Executer une transaction
    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        nonce++;
        (bool success, ) = dest.call{value: value}(data);
        require(success, "execution failed");
        emit Executed(dest, value, data);
    }

    // Executer un batch
    function executeBatch(
        address[] calldata dests,
        uint256[] calldata values,
        bytes[] calldata datas
    ) external onlyOwner {
        require(dests.length == values.length && dests.length == datas.length, "length mismatch");
        for (uint256 i = 0; i < dests.length; i++) {
            nonce++;
            (bool success, ) = dests[i].call{value: values[i]}(datas[i]);
            require(success, "batch call failed");
            emit Executed(dests[i], values[i], datas[i]);
        }
    }

    // Changer le owner (social recovery, etc.)
    function changeOwner(address newOwner) external onlyOwner {
        require(newOwner != address(0), "zero address");
        emit OwnerChanged(owner, newOwner);
        owner = newOwner;
    }

    receive() external payable {}
}
'''

# Deployer sur anvil
account, receipt = compile_and_deploy(w3, STANDALONE_ACCOUNT, deployer, deployer)

# Verifier le owner
print(f"Owner: {account.functions.owner().call()}")
print(f"Nonce: {account.functions.nonce().call()}")

# Envoyer de l ETH au smart account
tx = w3.eth.send_transaction({
    'from': deployer,
    'to': account.address,
    'value': w3.to_wei(1, 'ether')
})
w3.eth.wait_for_transaction_receipt(tx)
balance = w3.eth.get_balance(account.address)
print(f"Balance du smart account: {w3.from_wei(balance, 'ether')} ETH")


---

## 3. Paymaster

Un paymaster peut payer les frais de gas pour les utilisateurs.

In [3]:
# Verifying Paymaster (ERC-4337)
# Un paymaster peut sponsoriser les frais de gas.
# Ce contrat herite de BasePaymaster (account-abstraction SDK)
# et necessite un EntryPoint deploye.

PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@account-abstraction/contracts/core/BasePaymaster.sol";
import "@account-abstraction/contracts/core/Helpers.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

contract VerifyingPaymaster is BasePaymaster {
    using ECDSA for bytes32;

    address public verifyingSigner;

    constructor(IEntryPoint _entryPoint, address _verifyingSigner)
        BasePaymaster(_entryPoint, msg.sender)
    {
        verifyingSigner = _verifyingSigner;
    }

    // Validation du paymaster
    function _validatePaymasterUserOp(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash,
        uint256 maxCost
    ) internal override returns (bytes memory context, uint256 validationData) {
        (uint48 validUntil, uint48 validAfter, bytes calldata signature) =
            parsePaymasterAndData(userOp.paymasterAndData);

        bytes32 hash = keccak256(abi.encode(userOpHash, validUntil, validAfter));
        bytes32 ethSignedHash = MessageHashUtils.toEthSignedMessageHash(hash);

        if (verifyingSigner != ECDSA.recover(ethSignedHash, signature)) {
            return ("", _packValidationData(true, validUntil, validAfter));
        }

        return ("", _packValidationData(false, validUntil, validAfter));
    }

    function parsePaymasterAndData(bytes calldata paymasterAndData)
        internal pure
        returns (uint48 validUntil, uint48 validAfter, bytes calldata signature)
    {
        require(paymasterAndData.length >= 52, "invalid paymasterAndData");
        validUntil = uint48(bytes6(paymasterAndData[20:26]));
        validAfter = uint48(bytes6(paymasterAndData[26:32]));
        signature = paymasterAndData[32:];
    }
}
'''

# Compilation avec Foundry
abi_pm, bytecode_pm = forge_compile(PAYMASTER, "VerifyingPaymaster")
print(f"Compilation reussie ! ABI: {len(abi_pm)} fonctions/events")
print(f"Bytecode: {len(bytecode_pm)} caracteres")
print()

# Afficher les fonctions publiques
print("Fonctions et events du VerifyingPaymaster:")
for item in abi_pm:
    if item.get('type') == 'function':
        inputs = ', '.join(f"{i['type']} {i['name']}" for i in item.get('inputs', []))
        print(f"  function {item['name']}({inputs})")
    elif item.get('type') == 'event':
        print(f"  event {item['name']}")

print()
print("Note: deploiement impossible sans EntryPoint reel (interface ERC-165 requise).")
print("En production, le Paymaster est deploye apres l'EntryPoint.")


### Version standalone deployable

Version simplifiee d'un paymaster qui illustre le mecanisme de sponsoring
de gas sans dependre de l'infrastructure EntryPoint.

In [ ]:
# Version standalone du Paymaster (sans dependances ERC-4337)

STANDALONE_PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title StandalonePaymaster
/// @notice Demonstre le concept de sponsoring de gas.
/// Le signer autorise des operations en signant off-chain.
contract StandalonePaymaster {
    address public owner;
    address public verifyingSigner;
    mapping(address => uint256) public deposits;

    event Deposited(address indexed account, uint256 amount);
    event Withdrawn(address indexed account, uint256 amount);
    event GasSponsored(address indexed account, uint256 amount);

    constructor(address _signer) {
        owner = msg.sender;
        verifyingSigner = _signer;
    }

    // Deposer des fonds pour sponsoriser le gas
    function deposit() external payable {
        deposits[msg.sender] += msg.value;
        emit Deposited(msg.sender, msg.value);
    }

    // Retirer des fonds
    function withdraw(uint256 amount) external {
        require(deposits[msg.sender] >= amount, "insufficient deposit");
        deposits[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
        emit Withdrawn(msg.sender, amount);
    }

    // Simuler le sponsoring de gas
    function sponsorGas(address account, uint256 gasAmount) external {
        require(msg.sender == owner, "only owner");
        require(deposits[owner] >= gasAmount, "insufficient funds");
        deposits[owner] -= gasAmount;
        emit GasSponsored(account, gasAmount);
    }

    function getDeposit(address account) external view returns (uint256) {
        return deposits[account];
    }
}
'''

# Deployer
paymaster, receipt = compile_and_deploy(w3, STANDALONE_PAYMASTER, deployer, deployer)

# Deposer des fonds
tx = paymaster.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(5, 'ether')})
w3.eth.wait_for_transaction_receipt(tx)
deposit = paymaster.functions.getDeposit(deployer).call()
print(f"Deposit du paymaster: {w3.from_wei(deposit, 'ether')} ETH")

# Sponsoriser du gas pour un compte
user = w3.eth.accounts[1]
tx = paymaster.functions.sponsorGas(user, w3.to_wei(0.1, 'ether')).transact({'from': deployer})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Gas sponsorise pour {user[:10]}... : 0.1 ETH")
deposit_after = paymaster.functions.getDeposit(deployer).call()
print(f"Deposit restant: {w3.from_wei(deposit_after, 'ether')} ETH")


---

## 4. Flux de Transaction ERC-4337

In [4]:
# Flux ERC-4337
print("""
FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
""")


FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789



---

## 5. Exercices

In [ ]:
# Exercice: Social Recovery Account
# Version standalone (sans dependances ERC-4337) - compilation et deploiement reel

EXERCISE_RECOVERY = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title SocialRecoveryAccount
/// @notice Smart account avec recuperation sociale via des gardiens.
/// Si le owner perd sa cle, les gardiens peuvent voter pour changer le owner.
contract SocialRecoveryAccount {
    address public owner;
    address[] public guardians;
    mapping(address => bool) public isGuardian;
    uint256 public guardianCount;
    uint256 public threshold;
    uint256 public nonce;

    // Recovery state
    address public pendingNewOwner;
    uint256 public recoveryApprovals;
    mapping(address => bool) public hasApprovedRecovery;

    event GuardianAdded(address indexed guardian);
    event RecoveryStarted(address indexed newOwner);
    event RecoveryApproved(address indexed guardian);
    event RecoveryCompleted(address indexed newOwner);
    event Executed(address indexed dest, uint256 value);

    modifier onlyOwner() {
        require(msg.sender == owner, "not owner");
        _;
    }

    modifier onlyGuardian() {
        require(isGuardian[msg.sender], "not guardian");
        _;
    }

    constructor(
        address _owner,
        address[] memory _guardians,
        uint256 _threshold
    ) {
        require(_threshold <= _guardians.length, "threshold too high");
        require(_threshold > 0, "threshold must be positive");
        owner = _owner;
        threshold = _threshold;
        for (uint256 i = 0; i < _guardians.length; i++) {
            address g = _guardians[i];
            require(!isGuardian[g], "duplicate guardian");
            guardians.push(g);
            isGuardian[g] = true;
            guardianCount++;
            emit GuardianAdded(g);
        }
    }

    // Executer une transaction (owner seulement)
    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        nonce++;
        (bool success, ) = dest.call{value: value}(data);
        require(success, "execution failed");
        emit Executed(dest, value);
    }

    // Demarrer une recuperation (gardien seulement)
    function startRecovery(address newOwner) external onlyGuardian {
        require(pendingNewOwner == address(0), "recovery already pending");
        require(newOwner != address(0), "zero address");
        pendingNewOwner = newOwner;
        recoveryApprovals = 1;
        hasApprovedRecovery[msg.sender] = true;
        emit RecoveryStarted(newOwner);
    }

    // Approuver une recuperation (gardien seulement)
    function approveRecovery() external onlyGuardian {
        require(pendingNewOwner != address(0), "no recovery pending");
        require(!hasApprovedRecovery[msg.sender], "already approved");
        hasApprovedRecovery[msg.sender] = true;
        recoveryApprovals++;
        emit RecoveryApproved(msg.sender);

        if (recoveryApprovals >= threshold) {
            address newOwner = pendingNewOwner;
            // Reset recovery state
            pendingNewOwner = address(0);
            recoveryApprovals = 0;
            for (uint256 i = 0; i < guardians.length; i++) {
                hasApprovedRecovery[guardians[i]] = false;
            }
            owner = newOwner;
            emit RecoveryCompleted(newOwner);
        }
    }

    // Annuler la recuperation (owner seulement)
    function cancelRecovery() external onlyOwner {
        require(pendingNewOwner != address(0), "no recovery pending");
        pendingNewOwner = address(0);
        recoveryApprovals = 0;
        for (uint256 i = 0; i < guardians.length; i++) {
            hasApprovedRecovery[guardians[i]] = false;
        }
    }

    receive() external payable {}
}
'''

# Deployer le SocialRecoveryAccount
guardian1 = w3.eth.accounts[1]
guardian2 = w3.eth.accounts[2]
guardian3 = w3.eth.accounts[3]

recovery_account, receipt = compile_and_deploy(
    w3, EXERCISE_RECOVERY, deployer,
    deployer,                                    # owner
    [guardian1, guardian2, guardian3],             # guardians
    2                                            # threshold: 2 sur 3
)

print(f"Owner: {recovery_account.functions.owner().call()[:10]}...")
print(f"Guardian count: {recovery_account.functions.guardianCount().call()}")
print(f"Threshold: {recovery_account.functions.threshold().call()}")

# Tester la recuperation sociale
new_owner = w3.eth.accounts[4]
print(f"\n--- Demarrage recuperation vers {new_owner[:10]}... ---")

# Guardian 1 demarre la recuperation
tx = recovery_account.functions.startRecovery(new_owner).transact({'from': guardian1})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Guardian 1 a demarre la recuperation (1/{recovery_account.functions.threshold().call()})")

# Guardian 2 approuve -> threshold atteint
tx = recovery_account.functions.approveRecovery().transact({'from': guardian2})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Guardian 2 a approuve (2/{recovery_account.functions.threshold().call()})")

# Verifier le nouveau owner
current_owner = recovery_account.functions.owner().call()
print(f"\nNouveau owner: {current_owner[:10]}...")
assert current_owner == new_owner, "Le owner n'a pas change !"
print("Recuperation sociale reussie !")


---

## 6. Resume

| Concept | Description |
|---------|-------------|
| UserOperation | Structure de transaction ERC-4337 |
| EntryPoint | Singleton qui traite les UserOps |
| Smart Account | Wallet programmable |
| Paymaster | Sponsor de gas fees |
| Bundler | Empaqueteur de transactions |
| Social Recovery | Recuperation via gardiens |

---

**Notebook suivant** : [SC-11-LLM-Assisted](SC-11-LLM-Assisted.ipynb)

---

[<< DAO Governance](SC-9-DAO-Governance.ipynb) | [LLM Assisted >>](SC-11-LLM-Assisted.ipynb)